> ⚠️ **WARNING — UNDER CONSTRUCTION**
>
> This notebook is a work in progress. **Use at your own risk.**
> It may contain errors, incomplete sections, or incorrect results.
> No guarantees are made about its accuracy or completeness.

# Photometric Redshifts
## SED template fitting and machine-learning photo-z estimation

**Data:** Fully synthetic (self-contained, no download required)  
**Concept reference:** Bolzonella, Miralles & Pelló (2000), A&A 363, 476; Ilbert et al. (2006), A&A 457, 841

**Key tools:** `numpy`, `scipy`, `matplotlib`, `scikit-learn`, `astropy`

---

## Learning objectives

After this tutorial you will be able to:
1. Understand **why photometric redshifts are necessary** for large imaging surveys.
2. Build a library of **galaxy SED templates** (elliptical, spiral, starburst).
3. Implement a **template-fitting algorithm**: find $z$ and template type that minimise $\chi^2$.
4. Evaluate photo-z accuracy: **bias**, **$\sigma_{\rm NMAD}$**, and **catastrophic outlier rate**.
5. Compare template fitting to a **machine-learning approach** (k-nearest neighbours).

---

## 1. Theoretical background

### 1.1 Why photometric redshifts?

The largest existing and upcoming galaxy surveys — SDSS, DES, HSC, Euclid, LSST/Rubin — image billions of galaxies, but obtaining spectroscopic redshifts for even 1% of them requires enormous telescope time. For the remaining 99%, we must estimate redshifts from broadband photometry alone: these are **photometric redshifts** (photo-z).

A galaxy's SED (spectral energy distribution) contains spectral features — the 4000 Å break, the Balmer break, strong emission lines like H$\alpha$ and [OII] — that shift through the broadband filters as $z$ increases. By modelling how the observed colours $(u-g, g-r, r-i, i-z)$ depend on $z$, we can estimate the redshift from photometry alone.

### 1.2 The SED template fitting method

Given observed fluxes $f_\lambda^{\rm obs}$ in filters $b = 1, \ldots, N_b$ with uncertainties $\sigma_b$, and a library of SED templates $\{T_t(\lambda)\}$, the photo-z is found by minimising:

$$\chi^2(z, t) = \sum_{b=1}^{N_b} \frac{\left[f_b^{\rm obs} - a_{z,t}\, f_b^{\rm pred}(z, t)\right]^2}{\sigma_b^2}$$

where $f_b^{\rm pred}(z, t)$ is the predicted flux in band $b$ for template $t$ redshifted to $z$:

$$f_b^{\rm pred}(z, t) = \int_0^\infty T_t\!\left(\frac{\lambda}{1+z}\right) R_b(\lambda)\,\lambda\,d\lambda$$

and $a_{z,t}$ is an overall normalisation (galaxy distance/luminosity). The best-fit $\hat{z}$ and $\hat{t}$ are those minimising $\chi^2$ over the grid $(z, t)$.

References: **Bolzonella, Miralles & Pelló (2000)**, A&A 363, 476. [arXiv:astro-ph/0003380](https://arxiv.org/abs/astro-ph/0003380); **Ilbert et al. (2006)**, A&A 457, 841. [arXiv:astro-ph/0603217](https://arxiv.org/abs/astro-ph/0603217)

### 1.3 SED templates

The templates span the range of galaxy types observed in the Universe:
- **Elliptical (E):** old, passively evolving stellar populations. Red continuum peaking near 7000 Å. Strong 4000 Å break. No emission lines. Reference: Coleman, Wu & Weedman (1980), ApJS 43, 393.
- **Sa (early spiral):** mix of old bulge + young disk stars. Intermediate colour. Weak emission lines.
- **Sc (late spiral):** younger stellar populations. Bluer continuum. Moderate H$\alpha$ and [OII] emission.
- **Starburst (SB):** intense ongoing star formation. Very blue UV continuum. Strong emission lines: H$\alpha$ $\lambda6563$, [OII] $\lambda3727$, [OIII] $\lambda5007$. Reference: Kinney et al. (1996), ApJ 467, 38.

### 1.4 Photo-z quality metrics

Three standard metrics are used to evaluate photo-z performance (Laigle et al. 2016, COSMOS2015):

$$\Delta z = z_{\rm phot} - z_{\rm spec}$$

- **Bias:** $\langle \Delta z / (1 + z_{\rm spec}) \rangle$  
- **Scatter ($\sigma_{\rm NMAD}$):** $1.4826 \times {\rm MAD}\!\left(\Delta z / (1 + z_{\rm spec})\right)$  
  where MAD is the median absolute deviation. The factor 1.4826 makes it equivalent to $\sigma$ for a Gaussian.
- **Catastrophic outlier rate ($\eta$):** fraction of galaxies with $|\Delta z| / (1 + z_{\rm spec}) > 0.15$.

Reference: **Laigle et al.** (2016), ApJS 224, 24. [arXiv:1604.02350](https://arxiv.org/abs/1604.02350)

### 1.5 Machine-learning photo-z

An alternative to template fitting is supervised machine learning: train a model on galaxies with known spectroscopic redshifts (the **training set**) and predict $z_{\rm phot}$ for the rest.

The **k-nearest neighbours (k-NN)** regressor predicts the photo-z of a new galaxy as the average (or median) redshift of its $k$ nearest neighbours in colour space $(u-g, g-r, r-i, i-z)$. It is simple but effective when the training set is representative of the target population.

---

## 2. Setup and imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import AutoMinorLocator
import warnings
warnings.filterwarnings('ignore')

from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import median_abs_deviation
from scipy.optimize import minimize_scalar
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Reproducibility
np.random.seed(42)
rng = np.random.default_rng(42)

# Flat LCDM cosmology
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 11,
})

print('All imports successful.')

---

## 3. Building galaxy SED templates

We construct four simple galaxy SED templates over the wavelength range 900–11000 Å. Each template is a superposition of:
1. A **Planck (blackbody) continuum** representing the stellar population.
2. A **4000 Å break** (a spectral discontinuity due to absorption by metal-rich stellar atmospheres).
3. **Emission lines** (H$\alpha$, [OII], [OIII], H$\beta$) with strengths appropriate for each type.

References: Coleman, Wu & Weedman (1980), ApJS 43, 393; Kinney et al. (1996), ApJ 467, 38.

In [ ]:
# Wavelength grid: 900 -- 11000 Å (rest-frame)
LAM = np.linspace(900, 11000, 5000)   # Å

def planck_continuum(lam, T_eff):
    """Planck (blackbody) function as proxy for stellar continuum.

    Returns flux per unit wavelength (arbitrary units), normalised at 5500 Å.
    """
    # B_lambda proportional to lam^-5 / (exp(hc/lam kT) - 1)
    # hc/k = 1.4388e8 Å K
    hck = 1.4388e8   # Å K
    exponent = hck / (lam * T_eff)
    exponent = np.clip(exponent, 0, 500)
    flux = lam**(-5) / (np.exp(exponent) - 1)
    # Normalise at 5500 Å
    idx_norm = np.argmin(np.abs(lam - 5500))
    flux /= flux[idx_norm]
    return flux


def add_4000_break(lam, flux, break_strength=0.3):
    """Impose a 4000 Å break by multiplying flux blueward of 4000 Å.

    break_strength: fractional flux drop blueward of 4000 Å.
    Strong break (0.5) for ellipticals, weak break (0.1) for starbursts.
    """
    mask_blue = lam < 4000
    # Smooth transition using a sigmoid
    sigma = 80.0   # Å width of the break
    suppression = 1.0 - break_strength / (1.0 + np.exp((lam - 3950) / sigma))
    return flux * suppression


def add_emission_line(lam, flux, lam0, ew_rest, fwhm=15.0):
    """Add a Gaussian emission line at rest wavelength lam0.

    lam0    : rest-frame wavelength [Å]
    ew_rest : rest-frame equivalent width [Å] (positive for emission)
    fwhm    : line FWHM [Å]
    """
    sigma_line = fwhm / (2 * np.sqrt(2 * np.log(2)))
    # Find continuum level at lam0
    idx = np.argmin(np.abs(lam - lam0))
    cont = np.median(flux[max(0, idx-50):idx+50])
    line = cont * ew_rest / (np.sqrt(2*np.pi) * sigma_line)
    line_profile = line * np.exp(-0.5 * ((lam - lam0) / sigma_line)**2)
    return flux + line_profile


# Build the 4 SED templates
templates = {}

# --- E: Elliptical (old, passive, red) ---
# T_eff ~ 4500 K (K-giant dominated), strong 4000 Å break, no lines
f_E = planck_continuum(LAM, T_eff=4500)
f_E = add_4000_break(LAM, f_E, break_strength=0.55)
templates['E'] = f_E / np.trapezoid(f_E, LAM)

# --- Sa: Early spiral (old bulge + some young disk) ---
# Mix T=4500 K + T=8000 K continua, moderate break
f_Sa = 0.6 * planck_continuum(LAM, 4500) + 0.4 * planck_continuum(LAM, 8000)
f_Sa = add_4000_break(LAM, f_Sa, break_strength=0.30)
f_Sa = add_emission_line(LAM, f_Sa, lam0=6563, ew_rest=10,  fwhm=20)  # Hα
f_Sa = add_emission_line(LAM, f_Sa, lam0=3727, ew_rest=8,   fwhm=15)  # [OII]
templates['Sa'] = f_Sa / np.trapezoid(f_Sa, LAM)

# --- Sc: Late spiral (young disk dominated) ---
# T=12000 K dominated, weak break, moderate emission lines
f_Sc = 0.2 * planck_continuum(LAM, 4500) + 0.8 * planck_continuum(LAM, 12000)
f_Sc = add_4000_break(LAM, f_Sc, break_strength=0.12)
f_Sc = add_emission_line(LAM, f_Sc, lam0=6563, ew_rest=50,  fwhm=20)  # Hα
f_Sc = add_emission_line(LAM, f_Sc, lam0=4861, ew_rest=15,  fwhm=15)  # Hβ
f_Sc = add_emission_line(LAM, f_Sc, lam0=5007, ew_rest=30,  fwhm=15)  # [OIII]
f_Sc = add_emission_line(LAM, f_Sc, lam0=3727, ew_rest=35,  fwhm=15)  # [OII]
templates['Sc'] = f_Sc / np.trapezoid(f_Sc, LAM)

# --- SB: Starburst (very blue, strong lines) ---
# T=25000 K (O/B star dominated), very weak break, very strong lines
f_SB = planck_continuum(LAM, T_eff=25000)
f_SB = add_4000_break(LAM, f_SB, break_strength=0.05)
f_SB = add_emission_line(LAM, f_SB, lam0=6563, ew_rest=200, fwhm=25)  # Hα
f_SB = add_emission_line(LAM, f_SB, lam0=4861, ew_rest=50,  fwhm=20)  # Hβ
f_SB = add_emission_line(LAM, f_SB, lam0=5007, ew_rest=100, fwhm=20)  # [OIII]
f_SB = add_emission_line(LAM, f_SB, lam0=3727, ew_rest=120, fwhm=18)  # [OII]
templates['SB'] = f_SB / np.trapezoid(f_SB, LAM)

TEMP_NAMES  = list(templates.keys())
TEMP_COLORS = {'E': 'firebrick', 'Sa': 'darkorange',
               'Sc': 'steelblue', 'SB': 'royalblue'}

print('Templates built:', TEMP_NAMES)
print('Wavelength range: {:.0f} -- {:.0f} Å  ({} points)'.format(
    LAM.min(), LAM.max(), len(LAM)))

In [ ]:
# Plot the SED templates at rest frame
fig, ax = plt.subplots(figsize=(11, 5))

for name, flux in templates.items():
    ax.plot(LAM, flux / flux.max(), color=TEMP_COLORS[name],
            lw=2, label=name)

# Mark key spectral features
for lam0, label, ypos in [
    (3727, '[OII]', 0.85),
    (3970, 'CaII K', 0.75),
    (4000, '4000Å break', 0.65),
    (4861, 'Hβ', 0.85),
    (5007, '[OIII]', 0.90),
    (6563, 'Hα', 0.95),
]:
    ax.axvline(lam0, color='gray', lw=0.8, ls='--', alpha=0.6)
    ax.text(lam0 + 50, ypos, label, fontsize=8, color='gray', rotation=90, va='center')

ax.set_xlabel('Rest-frame wavelength $\\lambda$ [Å]')
ax.set_ylabel('Normalised flux density')
ax.set_title('Galaxy SED Templates (rest frame)')
ax.legend(loc='upper right')
ax.set_xlim(900, 11000)
ax.set_ylim(0, 1.05)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
plt.tight_layout()
plt.savefig('sed_templates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: sed_templates.png')

---

## 4. SDSS filter transmission curves

We approximate the five SDSS filters ($u, g, r, i, z$) as Gaussians centred at the effective wavelengths. This is sufficient for a photo-z demonstration; production codes use the measured filter curves.

| Filter | $\lambda_{\rm eff}$ [Å] | FWHM [Å] |
|--------|-------------------------|----------|
| $u$ | 3551 | 600 |
| $g$ | 4686 | 1370 |
| $r$ | 6166 | 1400 |
| $i$ | 7480 | 1530 |
| $z$ | 8932 | 1370 |

In [ ]:
FILTER_PARAMS = {
    'u': (3551,  600),
    'g': (4686, 1370),
    'r': (6166, 1400),
    'i': (7480, 1530),
    'z': (8932, 1370),
}
BANDS = list(FILTER_PARAMS.keys())
BAND_COLORS = {'u': 'purple', 'g': 'limegreen', 'r': 'firebrick',
               'i': 'darkorange', 'z': 'dimgray'}

def make_filter(lam, lam_eff, fwhm):
    """Gaussian approximation to a photometric filter."""
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    R = np.exp(-0.5 * ((lam - lam_eff) / sigma)**2)
    return R / np.trapezoid(R, lam)   # normalised

FILTERS = {b: make_filter(LAM, *FILTER_PARAMS[b]) for b in BANDS}

# Plot filters + example SED
fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(LAM, templates['E'] / templates['E'].max(),
                alpha=0.2, color='firebrick', label='E template')
ax.fill_between(LAM, templates['SB'] / templates['SB'].max(),
                alpha=0.2, color='royalblue', label='SB template')

for b in BANDS:
    R = FILTERS[b] / FILTERS[b].max()
    ax.plot(LAM, R, color=BAND_COLORS[b], lw=2, label=f'${b}$ filter')

ax.set_xlabel('Wavelength $\\lambda$ [Å]')
ax.set_ylabel('Normalised throughput / flux')
ax.set_title('SDSS Filter Curves (Gaussian approximation)')
ax.set_xlim(2500, 11000)
ax.legend(ncol=7, loc='upper left', fontsize=9)
ax.xaxis.set_minor_locator(AutoMinorLocator())
plt.tight_layout()
plt.savefig('sdss_filters.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 5. Synthetic photometry: colour–redshift tracks

For each template $t$ and redshift $z$, we compute the synthetic flux in each SDSS band:

$$f_b(z, t) = \int_0^\infty T_t\!\left(\frac{\lambda}{1+z}\right) R_b(\lambda)\,\lambda\,d\lambda$$

The resulting colour–redshift tracks show how the colours evolve with $z$ for each galaxy type. This is the information the photo-z code uses to estimate $z$.

In [ ]:
Z_GRID = np.arange(0.0, 1.51, 0.05)   # redshift grid for photo-z

def compute_synthetic_fluxes(lam, template_flux, filters, z):
    """
    Compute synthetic fluxes in all bands for a template at redshift z.

    Parameters
    ----------
    lam           : wavelength array [Å] (rest frame)
    template_flux : SED flux (same grid as lam)
    filters       : dict {band: response_array}
    z             : redshift

    Returns
    -------
    fluxes : dict {band: float}
    """
    # Redshifted SED: T(lambda/(1+z)) evaluated on observer-frame lambda grid
    lam_rest = lam / (1.0 + z)
    # Interpolate template at rest-frame wavelengths
    template_interp = np.interp(lam_rest, lam, template_flux,
                                 left=0.0, right=0.0)
    fluxes = {}
    for b, R in filters.items():
        # f_b = integral T(lambda/(1+z)) * R(lambda) * lambda dlambda
        integrand = template_interp * R * lam
        fluxes[b] = np.trapezoid(integrand, lam)
    return fluxes


# Precompute the grid of synthetic fluxes: shape (N_templates, N_z, N_bands)
N_t = len(TEMP_NAMES)
N_z = len(Z_GRID)
N_b = len(BANDS)

grid_fluxes = np.zeros((N_t, N_z, N_b))  # [template, z, band]

for it, tname in enumerate(TEMP_NAMES):
    for iz, z_ in enumerate(Z_GRID):
        f = compute_synthetic_fluxes(LAM, templates[tname], FILTERS, z_)
        for ib, b in enumerate(BANDS):
            grid_fluxes[it, iz, ib] = f[b]

print(f'Flux grid shape: {grid_fluxes.shape}  (templates × z × bands)')
print('Flux grid computed.')

In [ ]:
# Convert fluxes to AB magnitudes for plotting: m = -2.5*log10(f) + const
# We normalise at z=0 for each template to show colour tracks

# Colour tracks: g-r and r-i as function of z
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for it, tname in enumerate(TEMP_NAMES):
    f = grid_fluxes[it]   # shape (N_z, N_b)
    # Avoid log(0)
    f_safe = np.maximum(f, 1e-30)

    # Indices for g, r, i
    ig = BANDS.index('g')
    ir = BANDS.index('r')
    ii = BANDS.index('i')
    iu = BANDS.index('u')

    gr = -2.5 * np.log10(f_safe[:, ig] / f_safe[:, ir])
    ri = -2.5 * np.log10(f_safe[:, ir] / f_safe[:, ii])
    ug = -2.5 * np.log10(f_safe[:, iu] / f_safe[:, ig])

    axes[0].plot(Z_GRID, gr, color=TEMP_COLORS[tname], lw=2.5, label=tname)
    axes[1].plot(Z_GRID, ri, color=TEMP_COLORS[tname], lw=2.5, label=tname)

# Annotate key features
for ax, ylabel, title in zip(axes,
                               ['$(g - r)$', '$(r - i)$'],
                               ['$g - r$ vs redshift', '$r - i$ vs redshift']):
    ax.set_xlabel('Redshift $z$')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Colour–redshift tracks: {title}')
    ax.legend(title='Template')
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    # Mark 4000 Å break entering r band at z ~ 0.55
    ax.axvline(0.55, color='gray', ls=':', lw=1, alpha=0.6)
    ax.text(0.56, ax.get_ylim()[0] + 0.05 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
            '4000Å break\n→ $r$ band', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('color_redshift_tracks.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: color_redshift_tracks.png')

---

## 6. Generating a synthetic galaxy catalog

We generate $N = 600$ galaxies with known spectroscopic redshifts and template types, then add realistic photometric noise to simulate observations. Each galaxy is assigned:
- A random template type (E, Sa, Sc, SB) with weights $[0.25, 0.25, 0.30, 0.20]$
- A random redshift $z_{\rm spec}$ drawn from a distribution peaking at $z \approx 0.3$
- Photometric noise: Gaussian errors with $\sigma = 0.05$ mag (5% flux uncertainty)

In [ ]:
N_GAL     = 600
SNR_NOISE = 0.05   # fractional flux uncertainty → 5% noise

# Random template types
ttype_weights = [0.25, 0.25, 0.30, 0.20]  # E, Sa, Sc, SB
ttype_indices = rng.choice(len(TEMP_NAMES), size=N_GAL, p=ttype_weights)
ttype_true    = [TEMP_NAMES[i] for i in ttype_indices]

# Random spectroscopic redshifts (exponential distribution peaking at z=0.1)
z_true = rng.uniform(0.02, 0.80, N_GAL)
# Bias toward lower redshift (imaging surveys are flux limited)
z_true = z_true**(1.5) * 0.80   # adjust range
z_true = np.clip(z_true, 0.02, 0.79)

# Compute true fluxes for each galaxy
fluxes_true  = np.zeros((N_GAL, N_b))
for i in range(N_GAL):
    t_idx = ttype_indices[i]
    # Interpolate grid at the true z
    for ib in range(N_b):
        fluxes_true[i, ib] = np.interp(z_true[i],
                                         Z_GRID, grid_fluxes[t_idx, :, ib])

# Add noise: Gaussian in flux space, then convert to magnitudes
# sigma_flux = SNR_NOISE * flux
noise = rng.normal(0, SNR_NOISE, fluxes_true.shape) * fluxes_true
fluxes_obs = np.maximum(fluxes_true + noise, 1e-30)   # prevent negative fluxes

# Flux uncertainties
fluxes_err = SNR_NOISE * fluxes_true

# Convert to magnitudes (AB system, arbitrary zeropoint)
MAG_ZP = 25.0   # arbitrary AB zeropoint
mags_obs = MAG_ZP - 2.5 * np.log10(fluxes_obs)
mags_err = (2.5 / np.log(10)) * (fluxes_err / fluxes_obs)

print(f'Synthetic catalog: {N_GAL} galaxies')
print(f'z_spec range: {z_true.min():.3f} -- {z_true.max():.3f}')
print(f'Template type counts: E={ttype_true.count("E")}, Sa={ttype_true.count("Sa")},',
      f'Sc={ttype_true.count("Sc")}, SB={ttype_true.count("SB")}')
print(f'Magnitude range (r band): {mags_obs[:, BANDS.index("r")].min():.1f}',
      f'-- {mags_obs[:, BANDS.index("r")].max():.1f} mag')

---

## 7. Template-fitting photo-z algorithm

For each galaxy we minimise $\chi^2(z, t)$ over the redshift grid and all templates. The best-fit photo-z is the $z$ and template combination that gives the smallest $\chi^2$.

In [ ]:
def photoz_template_fit(flux_obs, flux_err, grid_fluxes, z_grid):
    """
    Find the best-fit photometric redshift and template via chi^2 minimisation.

    Parameters
    ----------
    flux_obs   : array (N_bands,)  — observed fluxes
    flux_err   : array (N_bands,)  — 1-sigma flux uncertainties
    grid_fluxes: array (N_templates, N_z, N_bands) — model flux grid
    z_grid     : array (N_z,)      — redshift grid

    Returns
    -------
    z_phot     : float — best-fit photometric redshift
    t_phot     : int   — best-fit template index
    chi2_min   : float — minimum chi^2
    """
    N_t, N_z, N_b = grid_fluxes.shape
    chi2 = np.full((N_t, N_z), np.inf)

    for it in range(N_t):
        for iz in range(N_z):
            f_model = grid_fluxes[it, iz, :]   # model fluxes
            # Analytic best-fit normalisation: a = sum(f_obs * f_mod / sigma^2) / sum(f_mod^2 / sigma^2)
            w = flux_err**(-2)
            numerator   = np.sum(w * flux_obs * f_model)
            denominator = np.sum(w * f_model**2)
            if denominator < 1e-30:
                continue
            a = numerator / denominator
            a = max(a, 1e-10)
            residuals = flux_obs - a * f_model
            chi2[it, iz] = np.sum(w * residuals**2)

    # Find minimum
    it_best, iz_best = np.unravel_index(np.argmin(chi2), chi2.shape)
    return z_grid[iz_best], it_best, chi2[it_best, iz_best]


print(f'Running template fitting on {N_GAL} galaxies...')
z_phot_tf    = np.zeros(N_GAL)
t_phot_tf    = np.zeros(N_GAL, dtype=int)
chi2_min_tf  = np.zeros(N_GAL)

for i in range(N_GAL):
    z_p, t_p, chi2_p = photoz_template_fit(
        fluxes_obs[i], fluxes_err[i], grid_fluxes, Z_GRID
    )
    z_phot_tf[i]   = z_p
    t_phot_tf[i]   = t_p
    chi2_min_tf[i] = chi2_p
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{N_GAL} done')

print('Template fitting complete.')

---

## 8. Evaluating template-fitting photo-z accuracy

We compute the three standard photo-z quality metrics.

In [ ]:
def photoz_metrics(z_spec, z_phot, label=''):
    """
    Compute standard photo-z quality metrics.

    Returns dict with bias, sigma_NMAD, outlier_rate.
    """
    dz     = z_phot - z_spec
    dz_norm = dz / (1.0 + z_spec)

    bias  = np.mean(dz_norm)
    # sigma_NMAD = 1.4826 * MAD(dz/(1+z))
    sigma_nmad = 1.4826 * median_abs_deviation(dz_norm)
    outlier_rate = np.mean(np.abs(dz_norm) > 0.15)

    if label:
        print(f'\n=== Photo-z metrics: {label} ===')
        print(f'  Bias         = {bias:+.5f}')
        print(f'  sigma_NMAD   = {sigma_nmad:.5f}')
        print(f'  Outlier rate = {outlier_rate:.3f}  ({100*outlier_rate:.1f}%)')
        print(f'  N galaxies   = {len(z_spec)}')

    return {'bias': bias, 'sigma_nmad': sigma_nmad,
            'outlier_rate': outlier_rate, 'N': len(z_spec)}


metrics_tf = photoz_metrics(z_true, z_phot_tf, label='Template fitting')

# --- Plot z_phot vs z_spec ---
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

ax = axes[0]
dz_norm_tf = (z_phot_tf - z_true) / (1 + z_true)
mask_ok_tf = np.abs(dz_norm_tf) <= 0.15
mask_out_tf = ~mask_ok_tf

ax.scatter(z_true[mask_ok_tf],  z_phot_tf[mask_ok_tf],
           c=[TEMP_COLORS[TEMP_NAMES[t]] for t in ttype_indices[mask_ok_tf]],
           s=15, alpha=0.6, label='Good fit')
ax.scatter(z_true[mask_out_tf], z_phot_tf[mask_out_tf],
           c='black', s=25, alpha=0.8, marker='x', label='Catastrophic')

z_diag = [0, 0.85]
ax.plot(z_diag, z_diag, 'k-', lw=1.5, label='1:1')
ax.plot(z_diag, [0, 0.85 * 1.15], 'k--', lw=1, alpha=0.5)
ax.plot(z_diag, [0, 0.85 * 0.85], 'k--', lw=1, alpha=0.5, label='$\\pm15\\%$')

ax.set_xlabel('$z_{\\rm spec}$')
ax.set_ylabel('$z_{\\rm phot}$  (template fitting)')
ax.set_title(f'Template Fitting\n$\\sigma_{{\\rm NMAD}}={metrics_tf["sigma_nmad"]:.3f}$, '
             f'outlier rate=${metrics_tf["outlier_rate"]*100:.1f}\\%$')
ax.legend(fontsize=10)
ax.set_xlim(0, 0.85)
ax.set_ylim(0, 0.85)

# Residuals
ax2 = axes[1]
ax2.hist(dz_norm_tf, bins=60, color='steelblue', edgecolor='white',
         linewidth=0.4, density=True)
ax2.axvline(0, color='k', lw=1.5)
ax2.axvline(metrics_tf['bias'], color='firebrick', lw=2,
            ls='--', label=f"Bias={metrics_tf['bias']:+.4f}")
ax2.axvspan(-0.15, 0.15, alpha=0.15, color='limegreen', label='$|\\Delta z/(1+z)| < 0.15$')
ax2.set_xlabel('$\\Delta z / (1+z_{\\rm spec})$')
ax2.set_ylabel('Probability density')
ax2.set_title('Residuals — Template Fitting')
ax2.legend()
ax2.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig('photoz_template_fit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: photoz_template_fit.png')

---

## 9. Machine-learning photo-z: k-NN and Random Forest

We now train a **k-nearest neighbours (k-NN) regressor** and a **Random Forest** on the galaxy colours. We split the catalog 50/50 into training and test sets.

The input features are four colours: $(u-g,\, g-r,\, r-i,\, i-z)$.

In [ ]:
# Build colour features from observed magnitudes
ib = {b: BANDS.index(b) for b in BANDS}
ug = mags_obs[:, ib['u']] - mags_obs[:, ib['g']]
gr = mags_obs[:, ib['g']] - mags_obs[:, ib['r']]
ri = mags_obs[:, ib['r']] - mags_obs[:, ib['i']]
iz = mags_obs[:, ib['i']] - mags_obs[:, ib['z']]

# Also add r-band magnitude as a luminosity proxy
X_all = np.column_stack([ug, gr, ri, iz, mags_obs[:, ib['r']]])
y_all = z_true

# 50/50 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.5, random_state=42
)
ttype_test = np.array([TEMP_NAMES.index(ttype_true[i])
                        for i in range(N_GAL)])[len(y_train):]

# Standardise features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set: {len(y_train)} galaxies')
print(f'Test set:     {len(y_test)} galaxies')

# --- k-NN Regressor ---
knn = KNeighborsRegressor(n_neighbors=7, weights='distance', metric='euclidean')
knn.fit(X_train_sc, y_train)
z_phot_knn = knn.predict(X_test_sc)
metrics_knn = photoz_metrics(y_test, z_phot_knn, label='k-NN (k=7)')

# --- Random Forest Regressor ---
rf = RandomForestRegressor(n_estimators=100, max_depth=12,
                            min_samples_leaf=5, random_state=42, n_jobs=2)
rf.fit(X_train_sc, y_train)
z_phot_rf = rf.predict(X_test_sc)
metrics_rf = photoz_metrics(y_test, z_phot_rf, label='Random Forest')

In [ ]:
# Also apply template fitting to the same test set
test_indices = np.arange(len(y_train), N_GAL)   # indices of test galaxies
z_phot_tf_test = z_phot_tf[test_indices]
metrics_tf_test = photoz_metrics(y_test, z_phot_tf_test, label='Template Fitting (test set)')

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True, sharex=True)

for ax, z_p, metrics, title in zip(
        axes,
        [z_phot_tf_test, z_phot_knn, z_phot_rf],
        [metrics_tf_test, metrics_knn, metrics_rf],
        ['Template Fitting', 'k-NN (k=7)', 'Random Forest']
):
    dz_n = (z_p - y_test) / (1 + y_test)
    c = np.abs(dz_n)
    sc = ax.scatter(y_test, z_p, c=c, cmap='RdYlGn_r', s=15, alpha=0.7,
                    vmin=0, vmax=0.3)
    z_d = [0, 0.85]
    ax.plot(z_d, z_d, 'k-', lw=1.5)
    ax.plot(z_d, [0, 0.85 * 1.15], 'k--', lw=0.8, alpha=0.5)
    ax.plot(z_d, [0, 0.85 * 0.85], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel('$z_{\\rm spec}$')
    ax.set_title(f'{title}\n'
                 f'$\\sigma_{{\\rm NMAD}}={metrics["sigma_nmad"]:.3f}$, '
                 f'$\\eta={metrics["outlier_rate"]*100:.1f}\\%$')
    ax.set_xlim(0, 0.85)
    ax.set_ylim(0, 0.85)

axes[0].set_ylabel('$z_{\\rm phot}$')
plt.colorbar(sc, ax=axes[-1], label='$|\\Delta z / (1+z)|$')
plt.suptitle('Photo-z Recovery — Three Methods Compared', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('photoz_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 10. Summary table

In [ ]:
print('=' * 72)
print('SUMMARY TABLE — Photo-z Performance (N={} test galaxies)'.format(len(y_test)))
print('=' * 72)
print(f"{'Method':<22} {'Bias':<14} {'sigma_NMAD':<14} {'Outlier rate':<14}")
print('-' * 72)
for method, m in [
    ('Template fitting',  metrics_tf_test),
    ('k-NN (k=7)',        metrics_knn),
    ('Random Forest',     metrics_rf),
]:
    print(f"  {method:<20} {m['bias']:>+8.5f}       "
          f"{m['sigma_nmad']:>8.5f}       "
          f"{m['outlier_rate']*100:>6.1f}%")
print('=' * 72)
print()
print('Definitions:')
print('  Bias         = mean(dz/(1+z_spec))')
print('  sigma_NMAD   = 1.4826 * MAD(dz/(1+z_spec))')
print('  Outlier rate = fraction with |dz/(1+z)| > 0.15')
print()
print('Literature reference (COSMOS2015, Laigle+2016, Le Phare):')
print('  sigma_NMAD ~ 0.007 -- 0.021 (multi-band, NUV to NIR)')
print('  Outlier rate ~ 1 -- 4%')
print('  (Our synthetic case is simpler with only 5 bands and 4 templates.)')

---

## 11. Exercises

**Exercise 1 — Prior on galaxy type**  
In the template-fitting code, the uniform prior over templates gives equal weight to E, Sa, Sc, and SB types. Modify the code to include a **luminosity-dependent prior** $p(t | m_r)$: at bright apparent magnitudes, ellipticals are more common. Use the red fractions measured in notebook N08. Does adding a prior improve $\sigma_{\rm NMAD}$?

**Exercise 2 — Effect of photometric noise on accuracy**  
Repeat the catalog generation with $\sigma_{\rm flux} = 0.02, 0.05, 0.10, 0.20$ (i.e. SNR 50, 20, 10, 5). For each noise level, compute $\sigma_{\rm NMAD}$ and the outlier rate for both template fitting and k-NN. Plot the degradation curves $\sigma_{\rm NMAD}(\sigma_{\rm flux})$. At what SNR does the photo-z quality degrade significantly?

**Exercise 3 — Bayesian photo-z and P(z)**  
Instead of a single point estimate, compute the posterior distribution $P(z | \mathbf{f})$ for each galaxy:
$$P(z | \mathbf{f}) \propto \sum_t \pi_t \exp(-\chi^2(z, t)/2)$$
Plot $P(z)$ for 10 representative galaxies. Compare the width of $P(z)$ with the actual $\sigma_{\rm NMAD}$. Which galaxies have the broadest $P(z)$, and why?

**Exercise 4 — Colour-redshift degeneracies**  
The most common catastrophic failures in template fitting occur when two different (template, $z$) combinations produce almost identical colours — the **colour-redshift degeneracy**. Find the 10 largest outliers in the template-fitting result and identify which templates and redshifts are confused. Plot their SEDs and predicted colours.

---

## Further reading

- **Bolzonella, Miralles & Pelló** (2000), A&A 363, 476 — BPZ template fitting. [arXiv:astro-ph/0003380](https://arxiv.org/abs/astro-ph/0003380)
- **Ilbert et al.** (2006), A&A 457, 841 — Le Phare photo-z code. [arXiv:astro-ph/0603217](https://arxiv.org/abs/astro-ph/0603217)
- **Laigle et al.** (2016), ApJS 224, 24 — COSMOS2015 photo-z catalog. [arXiv:1604.02350](https://arxiv.org/abs/1604.02350)
- **Coleman, Wu & Weedman** (1980), ApJS 43, 393 — galaxy SED templates.
- **Kinney et al.** (1996), ApJ 467, 38 — starburst galaxy UV–optical templates.